<a href="https://colab.research.google.com/github/telloudi-khadijaencg-alt/23009059-TELLOUDIKHADIJA-G3FINANCE/blob/main/notebooks/projet_F21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# F21 : Force du dollar

**Nom et prénom :** Telloudi Khadija    **Code Apogée :** 23009059    **Groupe :** G3FINANCE    **Filière :** S7 FIN    **Date :** 17/11/2026

**Dépôt GitHub :** https://github.com/telloudi-khadijaencg-alt/23009059-TELLOUDIKHADIJA-G3FINANCE

## 1. Sujet et problématique
- **Intérêt du sujet :** Le dollar influence les coûts d'importation, les exportations et la valeur des actifs internationaux, donc les entreprises exportatrices, les investisseurs et les banques centrales suivent son évolution. Un modèle qui anticipe son rendement du mois suivant pourrait aider à décider de la couverture du risque de change.
- **Problématique :** Les rendements passés du dollar permettent-ils de prévoir son rendement du mois suivant mieux qu'une règle naïve ?
- **Objectif mesurable :** RMSE sur le test inférieur d'au moins 2 % à celui de la moyenne d'apprentissage, test de 2020 à la dernière donnée.
- **Nature du problème :** régression sur série temporelle.
- **Algorithme principal envisagé :** n°2 (Ridge)

In [5]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sys
GRAINE = 42
np.random.seed(GRAINE)
print(sys.version, sklearn.__version__, pd.__version__)

3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0] 1.6.1 2.2.3


## 2. Jeu de données

- **Source :** FRED (Federal Reserve Bank of St. Louis), série DTWEXBGS, indice nominal large du dollar américain.
- **URL :** https://fred.stlouisfed.org/series/DTWEXBGS
- **Date d'extraction :** 17/11/2026
- **Licence :** (écris ce que dit la page FRED de la série ; si rien n'est indiqué, écris « non précisée »)
- **Taille et période :** ... observations, du ... au ..., fréquence quotidienne (jours ouvrés)
- **Unité d'observation :** un jour ouvré, puis un mois après agrégation en fin de mois
- **Variable cible :** rendement mensuel du mois suivant, en % (`cible`)
- **Adéquation :** la série couvre le dollar contre un panier large de monnaies pondéré par le commerce, ce qui permet d'étudier sa force globale.
- **Limites :** période courte (depuis 2006), une seule grande phase de dollar fort, chocs de 2008 et 2020 qui peuvent changer le régime, jours fériés manquants, série éventuellement révisée.

In [6]:
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DTWEXBGS"
Path("data/raw").mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(URL, index_col=0, parse_dates=True)
raw.to_csv("data/raw/DTWEXBGS.csv")
raw = pd.read_csv("data/raw/DTWEXBGS.csv", index_col=0, parse_dates=True)
raw.columns = ["indice"]
raw["indice"] = pd.to_numeric(raw["indice"], errors="coerce")

print(raw.shape, raw.index.min(), raw.index.max())
print(raw.isna().sum())
print(raw.describe())
print(raw.head())

(5445, 1) 2006-01-02 00:00:00 2026-11-13 00:00:00
indice    214
dtype: int64
            indice
count  5231.000000
mean    106.558713
std      12.366974
min      85.469200
25%      93.803650
50%     110.303400
75%     117.299750
max     130.041300
                    indice
observation_date          
2006-01-02        101.4155
2006-01-03        100.7558
2006-01-04        100.2288
2006-01-05        100.2992
2006-01-06        100.0241


| Variable | Description | Type | Unité | Rôle | Manquants (%) |
|---|---|---|---|---|---|
| indice | Niveau de l'indice large du dollar | numérique | points d'indice | brute | ... |
| r | Rendement mensuel = 100 × variation du log de l'indice fin de mois | numérique | % | explicative (retardée) | ... |
| r_lag1, r_lag2, r_lag3 | Rendements des 1, 2 et 3 mois précédents | numérique | % | explicative | ... |
| vol3 | Écart-type des rendements sur 3 mois, jusqu'à t | numérique | % | explicative | ... |
| mom12 | Somme des rendements sur 12 mois, jusqu'à t | numérique | % | explicative | ... |
| cible | Rendement du mois suivant (r décalé de −1) | numérique | % | cible | ... |

**Interprétation.** Le jeu contient 5 445 observations quotidiennes (jours ouvrés), du 2 janvier 2006 au 13 novembre 2026, soit environ 21 ans de données. La série comporte 214 valeurs manquantes (3,9 %), qui correspondent aux jours fériés où le marché est fermé et que FRED laisse vides. Je ne les supprime pas à ce stade : en prenant la dernière valeur de chaque mois, ces trous disparaissent, car chaque mois contient au moins un jour de cotation. L'indice varie entre 85,5 et 130,0 points, avec une moyenne de 106,6 et un écart-type de 12,4, ce qui montre que le dollar a connu de longues phases de hausse et de baisse sur la période. La période étant courte (environ 250 rendements mensuels), il faut rester prudent sur la complexité du modèle et sur la portée des conclusions.

In [7]:
m = raw["indice"].resample("ME").last().dropna()   # dernière valeur de chaque mois
m = m[m.index < "2026-11-01"]                       # on retire novembre 2026 (incomplet)

df = pd.DataFrame({"r": np.log(m).diff() * 100})    # rendement mensuel en %
for k in (1, 2, 3):
    df[f"r_lag{k}"] = df["r"].shift(k)              # rendements des mois précédents
df["vol3"] = df["r"].rolling(3).std()               # volatilité sur 3 mois, jusqu'à t
df["mom12"] = df["r"].rolling(12).sum()             # somme des rendements sur 12 mois
df["cible"] = df["r"].shift(-1)                     # rendement du mois SUIVANT
df = df.dropna()

print(df.shape)
print(df.head())
print(df.tail())

(237, 7)
                         r    r_lag1    r_lag2    r_lag3      vol3     mom12  \
observation_date                                                               
2007-01-31        0.956749  0.468264 -0.837463 -0.731785  0.927607 -1.486488   
2007-02-28       -0.743231  0.956749  0.468264 -0.837463  0.875240 -2.569477   
2007-03-31       -0.950942 -0.743231  0.956749  0.468264  1.046610 -4.309623   
2007-04-30       -1.550398 -0.950942 -0.743231  0.956749  0.419128 -3.425290   
2007-05-31       -0.544520 -1.550398 -0.950942 -0.743231  0.506017 -3.591374   

                     cible  
observation_date            
2007-01-31       -0.743231  
2007-02-28       -0.950942  
2007-03-31       -1.550398  
2007-04-30       -0.544520  
2007-05-31       -0.193356  
                         r    r_lag1    r_lag2    r_lag3      vol3     mom12  \
observation_date                                                               
2026-05-31        0.174532 -1.972480  2.690220 -0.065586  2.333778 

In [8]:
from google.colab import files
files.download("data/raw/DTWEXBGS.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>